# **Week 3: Sequence Models and Attention Mechanism**

Augment your sequence models using an attention mechanism, an algorithm that helps your model decide where to focus its attention given a sequence of inputs. Then, explore speech recognition and how to deal with audio data.

**Learning Objectives:**

- Describe a basic sequence-to-sequence model
- Compare and contrast several different algorithms for language translation
- Optimize beam search and analyze it for errors
- Use beam search to identify likely translations
- Apply BLEU score to machine-translated text
- Implement an attention model
- Train a trigger word detection model and make predictions
- Synthesize and process audio recordings to create train/dev datasets
- Structure a speech recognition project

---

## **Table of Contents**

---

## **Basic Models**

This section provides an introduction to **Sequence-to-Sequence (Seq2Seq)** architectures, which have revolutionized how we handle data that varies in length, such as sentences or audio clips. The core takeaway is the "Encoder-Decoder" framework: one part of the network understands the input, and the other part expresses the output.


### **High-level Summary**

We are looking at a fundamental shift in how neural networks handle data. In simpler models, we often had a fixed input size and a fixed output size. However, in tasks like **Machine Translation**, the input sentence "Jane visite l'Afrique en Septembre" (5 words in French) might result in an English output of 6 words. 

To bridge this gap, we use an **Encoder** to act as a "translator's brain," absorbing the full context of the input and condensing it into a dense mathematical vector. This vector is the "thought" that is then passed to the **Decoder**, which has the job of "speaking" that thought in the target language.

Interestingly, this logic extends beyond text. By swapping the RNN encoder for a **CNN**, we can effectively "translate" an image into a caption. The CNN sees a cat on a chair and turns those pixels into a feature vector; the RNN then takes that vector and generates the phrase "a cat sitting on a chair." The overarching magic here is that as long as we can turn an input into a representative vector, we can train an RNN to turn that vector into a sequence of meaningful words.


### **Key Takeaways**

* **The Encoder-Decoder Framework:**
    * **Encoder:** An RNN (often using GRU or LSTM units) processes the input sequence (e.g., a French sentence) and compresses it into a single representational vector.
    * **Decoder:** A separate RNN takes that vector as its starting point and generates the output sequence (e.g., the English translation) one word at a time until it hits an "End of Sequence" token.

<div align='center'>
<img src='assets/seq2seq-mt.png' width=500px>
</div>

* **Versatility of Inputs:** 
    * The "Encoder" doesn't have to be an RNN. For image captioning, we use a CNN (like a pre-trained AlexNet) to extract a feature vector from an image.
    * This feature vector is then fed into an RNN decoder to "describe" the image in natural language.

<div align='center'>
<img src='assets/seq2seq-ic.png' width=750px>
</div>

* **The Goal of "Best" vs. "Random":** * Unlike basic language models that might randomly sample words to generate creative text, Seq2Seq tasks like translation require finding the **most likely** sequence (the best translation), not just any sequence.

---

## **Picking the Most Likely Sentence**

This section builds upon the basic Seq2Seq architecture by framing it as a **Conditional Language Model** and explaining the search algorithms required to make it functional. The primary focus shifts from simply building the network to the computational challenge of finding the most accurate output.

### **High-level Summary**

We can see a fine line between the "generative" language models we often use for creative text and the "conditional" models needed for translation. In essence, the decoder is just a language model that has been "primed" with a specific thought from the encoder. Instead of asking the model, "What is a likely sentence to exist?", we are asking, "What is the most likely English version of *this* specific French idea?"

A significant portion of the discussion focuses on why we can't just take the easy way out with **Greedy Search**. Because language is a sequence where every choice impacts the future, picking the best word at the moment is like making a short-sighted chess move—it might look good now but ruin the "endgame" of the sentence. However, the search space for a perfect sentence is too massive for even the most powerful computers to check every possibility. This sets the stage for **Beam Search**, which acts as a middle ground: it’s more sophisticated than greedy search but far more efficient than checking every possible sentence in the dictionary.


### **Key Takeaways**

* **Machine Translation as a "Conditional" Model:** 
    * A standard language model estimates the probability of any given sentence: $P(y^{<1>}, \dots, y^{<T_y>})$.
    * A machine translation model estimates the probability of an English sentence **conditioned** on a specific French sentence: $P(y^{<1>}, \dots, y^{<T_y>} \mid x)$ and the goal is to find a $y$ to maximize this conditional probablity.
    * In a basic language model, the sequence starts with a vector of zeros. 
    * In Seq2Seq, the decoder starts with a **context vector** provided by the encoder, which represents the meaning of the input sentence.

<div align='center'>
<img src='assets/conditional_lm.png' width=750px>
</div>

* **Maximization vs. Random Sampling:**
    * Unlike generative language models (where we might sample words randomly for creativity), translation requires finding the single most likely sentence that maximizes the conditional probability.
* **The Failure of "Greedy Search":**
    * Greedy search picks the best word at step 1, then the best at step 2, and so on. 
    * This often fails because a "good" first word (e.g., "going") might lead the model down a path toward a suboptimal, verbose sentence, whereas a slightly "less likely" early word (e.g., "visiting") could lead to a better overall translation.
* **Search as an Optimization Problem:**
    * With "Exhaustiev Search" (or "brute force"), the total number of possible English sentences is astronomically large (e.g., $10,000^{10}$ for a 10-word sentence).
    * Since we cannot check every combination, we must use an approximate search algorithm like **Beam Search** to find a "good enough" maximum.


### **Bonus: What is Greedy Search Algorithm?**

Greedy search is a problem-solving strategy that makes the locally optimal choice at each step with the hope of finding a global optimum. It is "short-sighted" because it never reconsiders its past decisions, even if they lead to a dead end or a suboptimal outcome later. The term "greedy" describes the algorithm's single-minded focus on immediate, short-term gain without regard for future consequences or long-term efficiency. 

**How It Works:** At every stage of the search, the algorithm evaluates all available immediate options and picks the one that looks the best according to a specific criterion (like the lowest cost or highest immediate profit). It repeats this process until it reaches a goal or can no longer move forward.

**Common Applications:**
* Graph Algorithms: Dijkstra's Algorithm (shortest path), [Prim's](https://en.wikipedia.org/wiki/Greedy_algorithm) and Kruskal's algorithms (minimum spanning trees).
* Data Compression: Huffman Coding uses a greedy approach to build an optimal prefix tree.
* Natural Language Processing: In Large Language Models, it is used to generate text by always picking the most likely next word in a sequence.

---

## **Beam Search**

This section provides a detailed walkthrough of **Beam Search**, the industry-standard algorithm for finding the most likely output sequence in tasks like machine translation and speech recognition. While "Greedy Search" only looks at the single best word at each step, Beam Search maintains a set of high-probability candidates to ensure a more globally optimal result.

### **High-level Summary**

Beam Search is a more "farsighted" alternative to Greedy Search. In the running example of translating a French sentence, a greedy model might commit to a common first word like "in" and never look back. Beam Search, however, acts like a small team of scouts exploring the $B$ most promising paths simultaneously.

The beauty of the algorithm lies in its balance between **breadth** and **efficiency**. By evaluating $B \times \text{Vocabulary}$ options at every step, it allows the model to "change its mind." For example, even if "September" looked like a strong start in isolation, if it doesn't pair well with a second word, the algorithm will prune that path in favor of a stronger combination like "Jane visits."

Crucially, this isn't as computationally heavy as it sounds. While we consider 30,000 possibilities versus 10,000 in case with "Greedy Search" , we only ever "instantiate" $B$ copies of the network. This allows the decoder to process all 10,000 vocabulary options for each of the $B$ paths in parallel, making it a highly efficient way to find a "best" translation without checking the trillions of possible sentences in the entire language.

### **Key Takeaways**

* **The Beam Width ($B$):** 
    * This is the core parameter of the algorithm. It defines how many alternative sequences the model keeps in memory at any given time.
    * If $B=3$, the model tracks the 3 most likely partial sentences. If $B=1$, the algorithm simplifies back into a standard "Greedy Search".
* **Step-by-Step Expansion:**
    * **Step 1:** The model evaluates the probability of the first word across the entire vocabulary (e.g., 10,000 words) and picks the top $B$ candidates (e.g., "in", "Jane", "September").
    * **Step 2:** For *each* of those $B$ candidates, the model predicts the next 10,000 possible words. This creates $B \times \text{Vocabulary Size}$ (e.g., 30,000) total possibilities. The algorithm then narrows this list back down to the top $B$ most likely pairs of words.
* **Joint Probability Calculation:**
    * Beam Search doesn't just look for a likely second word; it maximizes the probability of the entire sequence.
    * It calculates this using the chain rule of probability: $P(y_1, y_2 \mid x) = P(y_1 \mid x) \times P(y_2 \mid x, y_1)$.
* **Pruning Candidates:**
    * The algorithm is "approximate" because it permanently drops candidates that fall out of the top $B$. For instance, if "September" is a likely first word but doesn't lead to a top-3 pair in Step 2, it is rejected and never considered again.
* **Process Termination:**
    * The search continues step-by-step (adding a 3rd word, a 4th, etc.) until the model predicts an `<EOS>` (End of Sentence) token for the candidates, resulting in a finalized, high-probability translation.

<figure style="text-align: center;">
  <img src="assets/beam_search.png" alt="Beam Search Diagram" style="width: 750px;">
  <figcaption>Schematic of Beam Search with B=2.
  Image by <a href="https://towardsdatascience.com/foundations-of-nlp-explained-visually-beam-search-how-it-works-1586b9849a24/">Ketan Doshi</a>.
  </figcaption>
</figure>

---

## **Refinements to Beam Search**

This section introduces essential refinements to the basic Beam Search algorithm, specifically addressing numerical stability and the bias toward short sequences. It also provides brief practical guidance on hyperparameter tuning for production and research environments.

### **Narrative Summary**

The focuse here is on how of making Beam Search work in the real world. The first major hurdle is purely mathematical: computers are bad at multiplying thousands of tiny decimals. By switching to **log space**, we transform a precarious multiplication task into a stable addition task.

The second hurdle is more "human". Without intervention, the model learns that "brevity is the soul of wit" simply because shorter sentences involve less math and thus higher probabilities. To force the model to consider longer, more complete translations, we apply **Length Normalization**. By averaging the score across the length of the sentence, we give longer translations a fair fighting chance against shorter ones.

Finally, the discussion settles on the practicalities of the **Beam Width**. Choosing $B$ is a classic engineering trade-off. While a $B$ of 3,000 might help a DL/AI practionar win a Kaggle competition or publish a research paper, the massive computational and memory cost often isn't worth it for a production app where users expect instant results. Most professional systems find their "sweet spot" around a beam width of 10, where the balance between quality and speed is most efficient.

### **Key Takeaways**

* **Log Probability Summation:**
    * The problem is multiplying many probabilities (all $< 1$) leads to **numerical underflow**, where the product becomes too small for a computer to store accurately.
    
    $$\arg\max_{y} = \prod_{t=1}^{T_y} P(y^{<t>} \mid x, y^{<1>}, y^{<2>}, \dots, y^{<t-1>})$$

    * To solve this, we maximize the **sum of log probabilities**: $\sum \log P(y^{<t>} \mid x, \dots)$ instead of maximizing the product. Since the logarithm is a *monotonically increasing* function, the sentence that maximizes the log-sum is the same one that maximizes the original product.

* **Length Normalization:**
    * Second problem is that standard beam search is biased toward short sentences. Because each additional word adds a negative value (log of a probability $< 1$) to the sum, shorter sentences naturally have "higher" (less negative) scores.
    * To resolve this, we divide the total score by the number of words ($T_y$) to get an average probability per word. A common heuristic uses a hyperparameter $\alpha$ (typically $0.7$):
    
    $$\frac{1}{T_y^\alpha} \sum \log P$$

    where $\alpha = 1$ is same as "full normalization" and $\alpha = 0$ is same as "no normalization".

* **The Beam Width ($B$) Trade-off:**
    * **Large $B$:** Explores more possibilities, generally leading to better results, but is slower and requires significantly more memory.
    * **Small $B$:** Faster and memory-efficient but prone to suboptimal "short-sighted" translations.
    * **Benchmarks:** Production systems often use $B \approx 10$. Research systems aiming for peak performance may use $B \approx 1,000$ to $3,000$, though gains show diminishing returns as $B$ increases.

* **Approximate vs. Exact Search:**
    * Unlike classic computer science algorithms like Breadth-First Search (BFS) or Depth-First Search (DFS), Beam Search is an **approximate** algorithm. It is much faster but is not guaranteed to find the absolute mathematical maximum.

---

## **Error Analysis in Beam Search**

This section introduces a targeted **Error Analysis** framework for debugging sequence-to-sequence models. When a model produces a poor translation ($\hat{y}$) compared to a human reference ($y^*$), it is often unclear whether the fault lies with the **RNN's training** (the model) or the **Beam Search's efficiency** (the search algorithm). This process provides a mathematical way to stop guessing and start fixing.

### **High-level Summary**

Sometimes we are tempted to simply throw more computation (larger beam width) or more data at a problem. By comparing the scores of the "ideal" human answer and the "flawed" AI answer, we can essentially peek into the model's brain to see where it went wrong. For example, score of ideal human-level answer is higher than the score of the model, the Beam Search is the blame but if the opposite holds true, RNN modeil has failed.

This systematic tallying allows a team to focus their energy. If most errors are "Model Errors," the team shouldn't waste time on search optimization. Instead, they should go back to the fundamental machine learning techniques to build a better underlying representation of language.

### **Key Takeaways**

* **The Diagnostic Tool:**
    * To attribute an error, we must compare two values using our RNN: the probability of the human translation $P(y^* \mid x)$ and the probability of the model's translation $P(\hat{y} \mid x)$.
* **Case 1: Beam Search is at Fault ($P(y^* \mid x) > P(\hat{y} \mid x)$)**
    * **Logic:** The RNN actually knew that the human translation was better (it gave $y^*$ a higher score), but Beam Search failed to find it.
    * **Fix:** Spend time increasing the beam width ($B$) or refining the search heuristic.
* **Case 2: The RNN Model is at Fault ($P(y^* \mid x) \le P(\hat{y} \mid x)$)**
    * **Logic:** The RNN incorrectly believes the bad translation is more likely than the human one. Even a perfect search algorithm would have chosen the bad translation because that's what the model "wants."
    * **Fix:** Focus on the neural network—get more training data, add regularization, or change the architecture.
* **Systematic Error Analysis:**
    * Instead of looking at one error, we go through our development (dev) set and tally the faults. If 80% of errors are Case 2, increasing the beam width is a waste of time. Here is how a typical Error Analysis Table would look. By filling this out for 30–50 "bad" translations in our dev set, we can clearly see whether we should be improving the RNN or improving Beam Search.

    | Sentence # | Input ($x$) | Human Translation ($y^*$) | Model Output ($\hat{y}$) | $P(y^* \mid x)$ | $P(\hat{y} \mid x)$ | **At Fault** |
    | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
    | 1 | *Jane visite l'Afrique...* | Jane visits Africa... | Jane visited Africa... | $2 \times 10^{-10}$ | $1 \times 10^{-10}$ | **Beam Search** |
    | 2 | *Il fait froid.* | It is cold. | It is freezing. | $5 \times 10^{-5}$ | $7 \times 10^{-5}$ | **RNN Model** |
    | 3 | *Où est le chat?* | Where is the cat? | Where is the car? | $1 \times 10^{-6}$ | $1.2 \times 10^{-6}$ | **RNN Model** |
    | 4 | *La pomme est rouge.* | The apple is red. | The red is apple. | $8 \times 10^{-8}$ | $2 \times 10^{-8}$ | **Beam Search** |


    After filling out the table, we calculate the percentages:

    * **If Beam Search Faults > 20-30%:** Our search algorithm is too "narrow." It's missing good answers the model already knows about.We should increase the Beam Width ($B$) or tune length normalization heuristic $\alpha$.
    * **If RNN Model Faults > 70-80%:** Our search is working fine, but the model's "logic" is wrong. It thinks bad English is better than good English. We should review our training data, check for overfitting, or try a Bidirectional RNN (to get better context before translating).


### **Bonus: Calculating Log-Probabilities in PyTorch**

To calculate the two scores for error analysis, we need to force the model to evaluate the "human" sentence $y^*$ word-by-word, even though the model didn't generate it.

The following Python function assumes we have a trained PyTorch `encoder` and `decoder` and have converted human sentence ($y^*$) and the model's output ($\hat{y}$) into their corresponding vocabulary indices:

```python
import torch
import torch.nn.functional as F

def calculate_sentence_score(sentence_indices, encoder_output, decoder, alpha=0.7):
    """
    Calculates the length-normalized log-probability score for a 
    specific sequence of word indices.
    """
    # 1. Initialize hidden state from encoder
    hidden = decoder.init_hidden(encoder_output)
    
    # Add <SOS> token to the start of the sequence for the decoder
    # Assuming sentence_indices is a list of word IDs [word1, word2, ..., <EOS>]
    input_token = torch.tensor([[SOS_TOKEN_ID]])
    
    total_log_prob = 0.0
    T_y = len(sentence_indices)
    
    with torch.no_grad(): # Set to inference mode
        for target_idx in sentence_indices:
            # 2. Forward pass through decoder
            output, hidden = decoder(input_token, hidden)
            
            # 3. Get log-probability of the specific target word
            # output is typically log_softmax or raw logits
            log_probs = F.log_softmax(output, dim=1)
            word_log_prob = log_probs[0, target_idx].item() # 0: batch index, .item(): converts tensor to scaler
            
            total_log_prob += word_log_prob
            
            # 4. Use the ACTUAL word from the sequence as next input 
            input_token = torch.tensor([[target_idx]])

    # 5. Apply Length Normalization
    normalized_score = total_log_prob / (T_y ** alpha)
    
    return normalized_score

# --- Usage in Error Analysis ---
score_human = calculate_sentence_score(y_star_indices, enc_out, decoder)
score_model = calculate_sentence_score(y_hat_indices, enc_out, decoder)

if score_human > score_model:
    print("Fault: Beam Search (The model preferred the human version but couldn't find it)")
else:
    print("Fault: RNN Model (The model actually thought the bad translation was better)")
```

---

## **Bleu Score**

This section introduces the **BLEU (Bilingual Evaluation Understudy) score**, the industry-standard metric for evaluating machine-generated text when multiple "correct" answers exist. In tasks like translation or image captioning, there isn't just one right answer, making traditional accuracy metrics useless. BLEU provides a single real-number score to automate evaluation.

### **High-level Summary**

A single French sentence like *"Le chat est sur le tapis"* can be translated as *"The cat is on the mat"* or *"There is a cat on the mat."* Since both are perfect, we can't use simple "correct/incorrect" logic.

The BLEU score solves this by looking for overlap. It asks: "How many of the phrases (n-grams) the machine produced appear in the human versions?" By using **Clipped Precision**, the algorithm prevents the model from gaming the system. If the word "the" appears twice in the reference, but the machine says it seven times, it only gets credit for two. Human-generated reference translations are typically included in dev set or test set for BLEU evaluation are 

To wrap up, we highlight the **Brevity Penalty**. This is the "anti-cheating" mechanism that ensures the model doesn't just output the one word it is most confident about to get a perfect precision score. While BLEU isn't perfect—it doesn't truly "understand" meaning—it serves as a powerful, automated tool for evaluating any system that generates text, from translation to describing a cat in an image.

### **Key Takeaways**

* **The "Understudy" Philosophy:**
    * Human evaluation is slow and expensive. BLEU acts as an "understudy" (substitute) for human judgment, providing a way to measure how close a machine translation is to human-provided references.
* **Modified Unigram Precision ($P_1$):**
    * Standard precision (checking if words appear in a reference) is easily "cheated" by repeating common words (e.g., "the the the...").
    * Modified Precision clips the count of each word. You only get credit for a word up to the maximum number of times it appears in the best reference sentence.
* **N-gram Overlap ($P_n$):**
    * To capture fluency and word order, BLEU looks at **bigrams** (pairs), **trigrams** (triplets), and **4-grams**. 
    * It calculates the clipped precision for these longer sequences to ensure the machine isn't just "word-vomiting" correct vocabulary in the wrong order.
* **Combined Score:**
    * The final BLEU score is typically a weighted average of the precision for $n=1$ through $n=4$, usually combined via an exponential of the average log-precisions. For details, see [here](https://en.wikipedia.org/wiki/BLEU).
* **Brevity Penalty (BP):**
    * Precision-based metrics naturally favor short outputs (it's easier to be 100% "precise" with a single word than a full sentence). 
    * The Brevity Penalty reduces the score if the machine's output is significantly shorter than the human references.
* **Impact on the Field:**
    * BLEU provided a "single real-number evaluation metric," allowing researchers to quickly compare models and iterate, which accelerated progress in NLP and Computer Vision.

---

## **Attention Model Intuition**

This section introduces the **Attention Model**, a pivotal evolution in Sequence-to-Sequence (Seq2Seq) architectures. While the basic Encoder-Decoder model struggles with "memorization" limits, the Attention Model allows the network to "focus" on specific parts of the input sentence while generating each word of the output, much like a human translator.

### **High-level Summary**

A fundamental flaw in the "Simple Encoder" approach is that it treats the neural network like it has a perfect, infinite memory, which isn't the case. As sentences grow longer, the model's ability to hold onto the beginning of the sentence while reaching the end diminishes, leading to a massive dip in translation quality.

To solve this, **Context Vector ($C_t$)** has been intriduced that changes at every step of the output. Instead of one static "thought" for the whole sentence, the model calculates a new "focus" for every word it writes. In our translation working example, when the model is trying to generate the English word "Africa," the **Attention Weights ($\alpha$)** will spike for the French word *"l'Afrique"*, effectively telling the decoder: "Ignore the rest of the sentence for a moment; look right here."

The technical "magic" happens through a **Bidirectional RNN**. By looking at the sentence from both directions, the model doesn't just see a word; it sees a word with its full surrounding context. The result is a model that doesn't just memorize—it "reads" as it "writes," maintaining high performance even as sentences stretch into long, complex paragraphs.

The formal mathematical derivation will be covered in next section. The aim here is build an intuition for relationship between the decoder's previous state $S_{t-1}$ and the encoder's features at every time step $t'$. The weight $\alpha_{t,t'}$ is a measure of how well the input at $t'$ matches what the decoder needs at step $t$ to produce the next correct word.

<figure style="text-align: center;">
  <img src="assets/attention_model_intuition.png" alt="Attention Model Intuition" style="width: 850px;">
  <figcaption>Schematic of Attention Model.</figcaption>
</figure>


### **Key Takeaways**

* **The "Bottleneck" Problem:** 
    * Traditional Encoder-Decoder models require the "Encoder" to compress an entire sentence into a single fixed-length vector. 
    * For long sentences ($>30$ or $40$ words), the BLEU score typically drops significantly because the network fails to memorize all the necessary details.

<figure style="text-align: center">
    <img src="assets/attention_model_performance.png" alt="Attention Model Performance" style='width: 750px;'>
    <figcaption>Figure borrowed from <a href="https://arxiv.org/pdf/1409.0473">Bahdanau et al. (2014).</a>
    </figcaption>
</figure>


* **Mimicing Human-Centric Translation:** 
    * Humans translate part-by-part rather than memorizing a 50-word sentence and regurgitating it from scratch. 
    * The Attention Model mimics this by allowing the Decoder to look at local windows of the input sequence.
* **The Architecture:**
    * **Bidirectional RNN (Encoder):** Processes the input to create a rich set of features for every word position, considering both previous and future context.
    * **Decoder States ($S_t$):** A separate RNN generates the output. At each step $t$, it maintains a hidden state $S_t$.
    * **Attention Weights ($\alpha_{t,t'}$):** These weights determine how much "attention" the model should pay to the $t'$-th input word when generating the $t$-th output word.
    * **Context Vector ($C$):** A weighted sum of the encoder features based on the attention weights, which is fed directly into the decoder unit.
* **Impact and Scope:** 
    * Proposed by Bahdanau et al., this seminal paper's influence has spread far beyond machine translation into almost every area of deep learning.



---

## **Attention Model**

This section dives into formulation of the **Attention Model**, moving from a high-level concept to the specific mathematical implementation. It explains how a decoder can selectively "attend" to specific parts of a source sequence by calculating a dynamic context vector at every step of the output.

### **High-level Summary**

The core of the implementation is the creation of a **Context Vector** that acts as a bridge between the encoder and decoder. Unlike the original Seq2Seq model, which forced the decoder to work from a single "memory" vector, this architecture allows the decoder to ask, "For the word I am about to write, which parts of the original sentence matter most?"

The "brain" that answers this question is a tiny, auxiliary neural network. It looks at what the decoder was just doing ($s^{\langle t-1 \rangle}$) and compares it to all the available input features ($a^{\langle t' \rangle}$). By training this via backpropagation, the model essentially learns its own alignment. We don't have to tell it that "visite" in French should map to "visits" in English; the gradient descent process discovers this relationship automatically.

The beauty of this system is best seen in **Attention Visualizations**. When we plot the weights $\alpha$ on a grid, we often see a bright diagonal line indicating a 1-to-1 word alignment, but we'll also see the model "look back" or "skip ahead" to handle grammatical differences between languages. It’s a computationally demanding process—running in quadratic time—but the gain in translation quality for long sentences is so significant that it has become a foundational pillar of modern deep learning.

### **Key Takeaways**

* **Feature Extraction (The Encoder):**
    * The model uses a **Bidirectional RNN** (usually LSTM or GRU) to process the input.
    * For each word position $t'$, the forward and backward hidden states are concatenated to form a single feature vector $a^{\langle t' \rangle}=[\overrightarrow{a}^{\langle t' \rangle}; \overleftarrow{a}^{\langle t' \rangle}]$. This vector contains information about the word and its surrounding context from both directions.

* **The Context Vector ($C$):**
    * Instead of one static vector for the whole sentence, the decoder calculates a unique context vector $c^{\langle t \rangle}$ for every output step $t$.
    * It is defined as a **weighted sum** of the encoder features:
        $$c^{\langle t \rangle} = \sum_{t'=1}^{T_x} \alpha^{\langle t, t' \rangle} a^{\langle t' \rangle}$$
    * The weights $\alpha^{\langle t, t' \rangle}$ represent the amount of "attention" the $t$-th output word should pay to the $t'$-th input word.

* **Calculating Attention Weights ($\alpha$):**
    * To ensure the weights are non-negative and sum to 1, a Softmax is applied to "energy" scores $e^{\langle t, t' \rangle}$:
        $$\alpha^{\langle t, t' \rangle} = \frac{\exp(e^{\langle t, t' \rangle})}{\sum_{k=1}^{T_x} \exp(e^{\langle t, k \rangle})}$$
    * The score $e^{\langle t, t' \rangle}$ is determined by a small, separate neural network (usually one hidden layer). Its inputs are the **previous decoder state** $s^{\langle t-1 \rangle}$ and the **encoder features** $a^{\langle t' \rangle}$ (See next bonus section for more details).

    <figure style='text-align: center'>
    <img src='assets/energy_score.png' alt='Energy Scores Neural Net' style='width=400px'>
    </figure>

* **Computational Cost:**
    * The algorithm has a quadratic cost ($O(T_x \times T_y)$). If the input has $T_x$ words and the output has $T_y$ words, there are $T_x \times T_y$ attention parameters to compute.
    * While expensive, this is generally acceptable for the typical lengths of sentences in machine translation.

* **Broad Applications:**
    * **Image Captioning:** Attention allows the model to look at specific regions of an image while generating corresponding words in a description.
    * **Date Normalization:** A common training exercise where the model learns to transform various date formats (e.g., "July 20th, 1969") into a standardized format (e.g., "1969-07-20").